In [0]:
%python
import os
import csv
import json
import zipfile
from datetime import datetime, timezone

base_path = "/Volumes/retail_dev/landing/landing_files"

archive_dir = f"{base_path}/archives"
customers_dir = f"{base_path}/customers"
orders_dir = f"{base_path}/orders"

for path in [archive_dir, customers_dir, orders_dir]:
    os.makedirs(path, exist_ok=True)

# Unique batch identifier prevents overwriting an earlier test batch.
batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
archive_name = f"retail_batch_{batch_id}.zip"
archive_path = f"{archive_dir}/{archive_name}"

customers = [
    {
        "customer_id": "C1001",
        "customer_name": "Anita Sharma",
        "email": "anita@example.com",
        "phone": "9876543210",
        "city": "Pune",
        "signup_date": "2025-01-15"
    },
    {
        "customer_id": "C1002",
        "customer_name": "Rahul Patil",
        "email": "invalid-email",
        "phone": "98765",
        "city": "Mumbai",
        "signup_date": "2025-02-20"
    },
    {
        "customer_id": "C1003",
        "customer_name": "",
        "email": "meera@example.com",
        "phone": "9123456780",
        "city": "Pune",
        "signup_date": "not-a-date"
    },
    {
        "customer_id": "C1001",
        "customer_name": "Anita Sharma",
        "email": "anita@example.com",
        "phone": "9876543210",
        "city": "Pune",
        "signup_date": "2025-01-15"
    },
    {
        "customer_id": "",
        "customer_name": "Karan Mehta",
        "email": "karan@example.com",
        "phone": "9988776655",
        "city": "Delhi",
        "signup_date": "2025-03-01"
    }
]

orders = [
    {
        "order_id": "O10001",
        "customer_id": "C1001",
        "order_amount": 1500.00,
        "order_status": "COMPLETED",
        "items": [
            {"product_id": "P101", "quantity": 2, "unit_price": 500.00},
            {"product_id": "P102", "quantity": 1, "unit_price": 500.00}
        ]
    },
    {
        "order_id": "O10002",
        "customer_id": "C1002",
        "order_amount": -250.00,
        "order_status": "COMPLETED",
        "items": [
            {"product_id": "P103", "quantity": -1, "unit_price": 250.00}
        ]
    },
    {
        "order_id": "O10003",
        "customer_id": "C9999",
        "order_amount": 800.00,
        "order_status": "UNKNOWN_STATUS",
        "items": [
            {"product_id": "P104", "quantity": 2, "unit_price": 400.00}
        ],
        "source_note": "Additional field for schema-evolution testing"
    },
    {
        "order_id": "O10004",
        "customer_id": "C1003",
        "order_amount": 600.00,
        "order_status": "PENDING",
        "items": [
            {"product_id": "P105", "quantity": 2, "unit_price": 300.00}
        ]
    }
]

# Write the source files into the ZIP without modifying their records.
import tempfile, shutil
with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp:
    tmp_path = tmp.name
with zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zf:
    with zf.open("customers.csv", "w") as raw_file:
        import io
        text_file = io.TextIOWrapper(
            raw_file, encoding="utf-8", newline=""
        )
        writer = csv.DictWriter(
            text_file, fieldnames=list(customers[0].keys())
        )
        writer.writeheader()
        writer.writerows(customers)
        text_file.flush()
        text_file.detach()

    zf.writestr(
        "orders.json",
        "\n".join(json.dumps(order) for order in orders) + "\n"
    )

shutil.copy2(tmp_path, archive_path)
os.remove(tmp_path)
print(f"Created archive: {archive_path}")
print(f"Customer records: {len(customers)}")
print(f"Order records: {len(orders)}")
print(f"Batch ID: {batch_id}")


In [0]:
DESCRIBE VOLUME retail_dev.landing.landing_files;

In [0]:
%python
import os
import zipfile

archive_dir = f"{base_path}/archives"

for filename in sorted(os.listdir(archive_dir)):
    if filename.endswith(".zip"):
        archive_path = f"{archive_dir}/{filename}"

        with zipfile.ZipFile(archive_path, "r") as zf:
            print(f"\nArchive: {filename}")
            print(zf.namelist())

In [0]:
%python

import os
import zipfile

base_path = "/Volumes/retail_dev/landing/landing_files"
archive_dir = f"{base_path}/archives"

for filename in sorted(os.listdir(archive_dir)):
    if not filename.lower().endswith(".zip"):
        continue

    archive_path = f"{archive_dir}/{filename}"

    print(f"\nArchive: {filename}")

    try:
        print("File size:", os.path.getsize(archive_path), "bytes")

        with open(archive_path, "rb") as f:
            print("First 8 bytes:", f.read(8))

        with zipfile.ZipFile(archive_path, "r") as zf:
            print("ZIP valid:", zf.testzip() is None)
            print("Contents:", zf.namelist())

    except Exception as e:
        print("ERROR:", type(e).__name__, str(e))

In [0]:
%python
base_path = "/Volumes/retail_dev/landing/landing_files"

display(dbutils.fs.ls(f"{base_path}/archives"))

In [0]:
%python

import os
import zipfile
import shutil

base_path = "/Volumes/retail_dev/landing/landing_files"
archive_dir = f"{base_path}/archives"

# 1. Find the ZIP archives.
zip_files = sorted(
    f for f in os.listdir(archive_dir)
    if f.lower().endswith(".zip")
)

if len(zip_files) != 1:
    raise ValueError(
        f"Expected exactly one ZIP archive; found {len(zip_files)}"
    )

archive_name = zip_files[0]
archive_path = f"{archive_dir}/{archive_name}"
batch_id = os.path.splitext(archive_name)[0]

# 2. Define the source-to-landing routing.
landing_dirs = {
    "customers.csv": f"{base_path}/customers",
    "orders.json": f"{base_path}/orders"
}

# Use unique destination filenames for this archive.
destination_paths = {
    member: f"{landing_dirs[member]}/{batch_id}_{member}"
    for member in landing_dirs
}

# 3. Validate the archive before extracting.
with zipfile.ZipFile(archive_path, "r") as zf:
    members = zf.namelist()

    missing = set(landing_dirs) - set(members)
    if missing:
        raise ValueError(f"Required files missing from ZIP: {missing}")

    bad_member = zf.testzip()
    if bad_member is not None:
        raise ValueError(f"Corrupt ZIP member: {bad_member}")

    # 4. Prevent overwriting files from an earlier run.
    existing = [
        path for path in destination_paths.values()
        if os.path.exists(path)
    ]

    if existing:
        raise FileExistsError(
            f"Output files already exist; refusing to overwrite: {existing}"
        )

    # 5. Extract to a staging directory first.
    staging_dir = f"{base_path}/staging/{batch_id}"
    os.makedirs(staging_dir, exist_ok=True)

    try:
        for member in landing_dirs:
            with zf.open(member, "r") as source:
                staging_path = f"{staging_dir}/{member}"
                with open(staging_path, "wb") as target:
                    shutil.copyfileobj(source, target)

        # 6. Publish only after both files have been staged.
        published = []

        try:
            for member, destination in destination_paths.items():
                os.makedirs(os.path.dirname(destination), exist_ok=True)
                os.replace(
                    f"{staging_dir}/{member}",
                    destination
                )
                published.append(destination)

        except Exception:
            # Best-effort cleanup if publishing fails.
            for path in published:
                if os.path.exists(path):
                    os.remove(path)
            raise

    finally:
        if os.path.exists(staging_dir):
            shutil.rmtree(staging_dir)

print(f"Archive processed: {archive_name}")
for member, destination in destination_paths.items():
    print(f"{member} -> {destination}")
